#### Required Libraries Import

Is cell me project ke liye required libraries import ki gayi hain. Python me **import** ka use kisi external package ya module ki functionality ko apne program me use karne ke liye hota hai.

**Imports ka explanation:**

- **StateGraph**
  - Ye LangGraph ka main class hai.
  - Iski help se hum workflow (graph) banate hain jisme multiple nodes aur unke beech connections define hote hain.
  - Example:
    - Pehle Joke Generate hoga.
    - Fir us Joke ki Explanation Generate hogi.

- **START aur END**
  - Ye predefined nodes hain.
  - START workflow ka beginning point hota hai.
  - END workflow complete hone ko represent karta hai.

- **TypedDict**
  - TypedDict ek typed dictionary hoti hai.
  - Iska use state ke structure ko define karne ke liye hota hai.
  - Matlab state me kaun-kaun si keys hongi aur unka datatype kya hoga.

- **ChatOpenAI**
  - Ye LangChain ka OpenAI Chat Model wrapper hai.
  - Iski help se hum GPT model ko prompts bhej kar response generate karte hain.

- **load_dotenv()**
  - Ye .env file ke variables ko load karta hai.
  - Jaise OpenAI API Key.

- **InMemorySaver**
  - Ye workflow ki state ko RAM me temporarily store karta hai.
  - Isse conversation ya workflow ka current progress save rehta hai.

**Summary**

Ye cell sirf project ke liye required tools aur classes ko import karta hai.

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain_groq import ChatGroq

d:\ML Projects\Langraph\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
load_dotenv()

llm = ChatGroq(model="llama-3.1-8b-instant", api_key=os.getenv("GROQ_API-KEY"))


NameError: name 'os' is not defined

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

### Workflow Execution with Configuration

Is cell me hum workflow ko **execute (run)** kar rahe hain. Yaha ek **configuration object** banaya gaya hai aur uske saath input data pass karke workflow ko invoke kiya gaya hai.

### `config1`

```python
config1 = {"configurable": {"thread_id": "1"}}
```

Ye ek **configuration dictionary** hai jo workflow ko additional settings provide karti hai.

#### `configurable`

`configurable` ek special key hai jiske andar workflow ki runtime settings di jaati hain.

#### `thread_id`

`thread_id` ek unique identifier hota hai jo ek specific workflow session ya conversation ko identify karta hai.

Iska fayda:
- Har user/session ki state alag maintain hoti hai.
- Agar workflow me memory use ho rahi hai (jaise `InMemorySaver`), to same `thread_id` ke saath us session ki state retrieve ki ja sakti hai.

**Example:**

```python
thread_id = "1"
```

Matlab ye execution **Thread 1** ke naam se save hogi.

Agar dusra user aaye:

```python
thread_id = "2"
```

to uski memory completely alag hogi.

---

### `workflow.invoke()`

```python
workflow.invoke({'topic':'pizza'}, config=config1)
```

`invoke()` workflow ko execute karne ka method hai.

Isme do cheezein pass ki gayi hain:

#### 1. Input State

```python
{'topic':'pizza'}
```

Ye initial state hai jo workflow ke first node ko milegi.

Is example me:

```
topic = "pizza"
```

Ab pehla node isi topic ke basis par joke generate karega.

---

#### 2. Configuration

```python
config=config1
```

Yaha workflow ko bataya ja raha hai ki execution **Thread ID = 1** ke under run hogi.

---

### Execution Flow

```
Input

topic = "pizza"

        │
        ▼

Generate Joke Node

        │
        ▼

Generate Explanation Node

        │
        ▼

Final State Return
```

---

### Expected Output

Workflow complete hone ke baad final state kuch is tarah hogi:

```python
{
    "topic": "pizza",
    "joke": "...Generated Joke...",
    "explanation": "...Generated Explanation..."
}
```

---

### Summary

- `config1` workflow ki runtime configuration define karta hai.
- `thread_id` ek unique session identifier hai jo memory ko manage karne me help karta hai.
- `workflow.invoke()` workflow ko start karta hai.
- Input state me `"pizza"` topic diya gaya hai.
- Workflow sequentially sabhi nodes execute karke final updated state return karta hai.

In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

NameError: name 'llm' is not defined

### Viewing Current State and State History

Is cell me hum workflow ki **current state** aur **poori execution history** dekh rahe hain. Ye debugging aur workflow ko samajhne ke liye bahut useful methods hain.

---

### `workflow.get_state(config1)`

```python
workflow.get_state(config1)
```

`get_state()` method kisi specific **thread/session** ki **latest (current) state** return karta hai.

Yaha `config1` pass kiya gaya hai jisme:

```python
thread_id = "1"
```

Isliye LangGraph **Thread 1** ki latest state fetch karega.

Output me `StateSnapshot` object milta hai.

Example:

```python
StateSnapshot(
    values={
        "topic":"pizza",
        "joke":"Why did the pizza go to the doctor?",
        "explanation":"..."
    }
)
```

Yaha `values` ke andar workflow ki current state stored hoti hai.

Matlab is time workflow ke paas ye saari information available hai.

---

### `StateSnapshot`

`StateSnapshot` ek object hai jo workflow ki kisi particular moment ki state ko represent karta hai.

Iske andar generally ye information hoti hai:

- Current Values
- Next Node
- Configuration
- Metadata
- Checkpoint Information

Simple words me bole to **StateSnapshot workflow ki memory ka photo (snapshot)** hota hai.

---

### `workflow.get_state_history(config1)`

```python
workflow.get_state_history(config1)
```

Ye method workflow ki **complete state history** return karta hai.

Matlab execution ke dauran har checkpoint par state kya thi, wo sab mil jata hai.

Ye directly ek iterator return karta hai.

Isliye usko list me convert kiya gaya hai.

```python
list(...)
```

---

### `list()`

Python ka `list()` function kisi iterator ya generator ko normal list me convert karta hai.

Agar hum sirf likhte:

```python
workflow.get_state_history(config1)
```

to iterator milta.

Lekin

```python
list(workflow.get_state_history(config1))
```

likhne se saare snapshots ek saath list ke form me mil jate hain.

---

### State History ka Flow

Output me snapshots reverse order me dikhte hain.

#### Initial State

```python
{}
```

Workflow abhi start bhi nahi hua tha.

↓

#### Input State

```python
{
    "topic":"pizza"
}
```

User ne topic diya.

↓

#### Joke Generated

```python
{
    "topic":"pizza",
    "joke":"Why did the pizza go to the doctor?"
}
```

Generate Joke node execute hua.

↓

#### Final State

```python
{
    "topic":"pizza",
    "joke":"Why did the pizza go to the doctor?",
    "explanation":"This joke..."
}
```

Generate Explanation node bhi complete ho gaya.

---

### Is Cell ka Purpose

Is cell ka purpose workflow ki **memory inspect karna** hai.

- `get_state()` → Current/latest state dikhata hai.
- `get_state_history()` → Workflow ki complete execution history dikhata hai.
- `StateSnapshot` → Kisi ek point par workflow ki memory ko represent karta hai.
- `list()` → History ke saare snapshots ek list ke form me display karta hai.

Ye methods debugging, testing aur workflow execution ko step-by-step samajhne ke liye bahut useful hote hain.

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f177f34-a1fc-67a4-8000-b20899a251c1'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-07-04T21:57:02.955101+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f177f34-a1f7-66fa-bfff-2a664cd174cc'}}, tasks=(PregelTask(id='40145101-d7ee-f78b-a935-98fde10ec569', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error='NameError("name \'llm\' is not defined")', interrupts=(), state=None, result=None),), interrupts=())

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f177f34-a1fc-67a4-8000-b20899a251c1'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-07-04T21:57:02.955101+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f177f34-a1f7-66fa-bfff-2a664cd174cc'}}, tasks=(PregelTask(id='40145101-d7ee-f78b-a935-98fde10ec569', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error='NameError("name \'llm\' is not defined")', interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f177f34-a1f7-66fa-bfff-2a664cd174cc'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-07-04T21:57:02.953036+00:00', parent_config=None, tasks=(PregelTask(id='5c9c7be6-7016-ca9e-6817-02ed0

### Executing Workflow with a Different Thread ID

Is cell me hum workflow ko **ek naye Thread ID (`2`)** ke saath execute kar rahe hain. Iska purpose ye dikhana hai ki LangGraph har **thread ki memory alag-alag maintain karta hai**.

---

### `config2`

```python
config2 = {"configurable": {"thread_id": "2"}}
```

Yaha ek naya configuration object banaya gaya hai jisme:

```python
thread_id = "2"
```

diya gaya hai.

Ye batata hai ki ab jo bhi workflow execute hoga, uski state **Thread 2** ke andar save hogi.

---

### Thread ID kya hota hai?

`thread_id` ek **unique identifier** hota hai jo har workflow session ko identify karta hai.

Iski wajah se multiple workflows ek saath chal sakte hain bina ek dusre ki state ko overwrite kiye.

Example:

```
Thread 1
Topic → Pizza

Thread 2
Topic → Pasta
```

Dono ki memory completely alag hogi.

---

### `workflow.invoke()`

```python
workflow.invoke({"topic":"pasta"}, config=config2)
```

Yaha workflow ko execute kiya gaya hai.

Input state:

```python
{
    "topic":"pasta"
}
```

Ab workflow pizza ki jagah **pasta** ke upar joke aur explanation generate karega.

Execution Flow:

```
Topic

↓

Generate Joke

↓

Generate Explanation

↓

Final State
```

---

### Output

Output me hume kuch is type ka result milta hai:

```python
{
    "topic":"pasta",
    "joke":"...",
    "explanation":"..."
}
```

Ye state **Thread 2** ke andar save hoti hai.

---

### `workflow.get_state(config1)`

```python
workflow.get_state(config1)
```

Ab hum dubara **Thread 1** ki state retrieve kar rahe hain.

Dhyan dene wali baat ye hai ki humne abhi Thread 2 run kiya tha, fir bhi Thread 1 ki state change nahi hui.

Output me fir bhi milega:

```python
{
    "topic":"pizza",
    "joke":"...",
    "explanation":"..."
}
```

Ye prove karta hai ki har thread apni alag memory maintain karta hai.

---

### `workflow.get_state_history(config1)`

```python
list(workflow.get_state_history(config1))
```

Ye method Thread 1 ki complete execution history return karta hai.

Yaha bhi sirf **Pizza workflow** ki history milegi.

Isme Pasta wala execution include nahi hoga kyunki wo **Thread 2** me execute hua tha.

---

### Memory Isolation

Is example me do independent workflow sessions create hue hain.

```
Thread 1

Topic → Pizza
Joke → Pizza Joke
Explanation → Pizza Explanation
```

```
Thread 2

Topic → Pasta
Joke → Pasta Joke
Explanation → Pasta Explanation
```

Dono threads ki memory completely independent hai.

---

### Is Cell ka Purpose

Is cell ka main objective ye demonstrate karna hai ki LangGraph me **Thread ID** ka use karke multiple independent workflow sessions maintain kiye ja sakte hain.

- `thread_id = "2"` ek naya workflow session create karta hai.
- `workflow.invoke()` naye thread me workflow execute karta hai.
- `get_state(config1)` se verify hota hai ki pehle wale thread ki state unchanged hai.
- `get_state_history(config1)` sirf Thread 1 ki history return karta hai.

Ye feature real-world applications jaise **chatbots, customer support systems aur multi-user applications** me bahut useful hota hai, jahan har user ki conversation aur memory alag maintain karni hoti hai.

In [10]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

NameError: name 'llm' is not defined

In [108]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!', 'explanation': 'This joke plays on the double meaning of the word "saucy." In one sense, "saucy" can mean bold, impertinent, or sassy. But in the context of a pizza, "saucy" refers to the tomato sauce typically used as a base on pizzas. So when the pizza went to the doctor because it was feeling "saucy," it implies that the pizza was not feeling well due to too much sauce, rather than being bold or sassy. The humor comes from the unexpected twist on the word\'s meaning.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-93a2-6a08-8002-395e36be0f5e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:56:42.071296+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7a2f-60ea-8001-4ac26c539f8d'}}, tasks=(), inte

In [109]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!', 'explanation': 'This joke plays on the double meaning of the word "saucy." In one sense, "saucy" can mean bold, impertinent, or sassy. But in the context of a pizza, "saucy" refers to the tomato sauce typically used as a base on pizzas. So when the pizza went to the doctor because it was feeling "saucy," it implies that the pizza was not feeling well due to too much sauce, rather than being bold or sassy. The humor comes from the unexpected twist on the word\'s meaning.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-93a2-6a08-8002-395e36be0f5e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:56:42.071296+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7a2f-60ea-8001-4ac26c539f8d'}}, tasks=(), int

### Time Travel

### Time Travel in LangGraph

Is cell me hum **LangGraph ke Time Travel feature** ka use kar rahe hain. Time Travel ka matlab hai workflow ke kisi **purane checkpoint (saved state)** par wapas jana aur wahin se workflow ko dobara continue karna.

Ye feature debugging, testing aur alternative execution paths ko explore karne ke liye bahut useful hota hai.

---

### `checkpoint_id`

```python
workflow.get_state({
    "configurable": {
        "thread_id": "1",
        "checkpoint_id": "..."
    }
})
```

Yaha `checkpoint_id` pass kiya gaya hai.

**Checkpoint** ek saved snapshot hota hai jo workflow execution ke dauran automatically create hota hai.

Har checkpoint workflow ki us time ki complete state ko store karta hai.

Example:

```
START

↓

Topic = Pizza

↓

Joke Generated

↓

Explanation Generated
```

In teeno steps ke alag-alag checkpoints save ho sakte hain.

---

### `workflow.get_state()`

```python
workflow.get_state(...)
```

Ye method given `thread_id` aur `checkpoint_id` ke according workflow ki state retrieve karta hai.

Output:

```python
StateSnapshot(
    values={
        "topic":"pizza"
    },
    next=("generate_joke",)
)
```

Yaha dekh sakte hain ki workflow us checkpoint par sirf topic tak pahucha tha aur next node `generate_joke` execute hona baaki tha.

---

### `next`

Output me

```python
next=("generate_joke",)
```

dikhta hai.

Ye batata hai ki agar workflow isi checkpoint se continue kiya gaya to sabse pehle **Generate Joke** node execute hoga.

---

### `workflow.invoke(None, config)`

```python
workflow.invoke(
    None,
    {
        "configurable":{
            "thread_id":"1",
            "checkpoint_id":"..."
        }
    }
)
```

Normally `invoke()` me input state pass ki jaati hai.

Lekin yaha

```python
None
```

pass kiya gaya hai.

Iska matlab hai:

> Koi nayi input mat do, workflow ko selected checkpoint se hi continue karo.

Workflow us saved state ko load karega aur wahi se remaining nodes execute karega.

---

### Is Example me kya hua?

Checkpoint par state thi:

```python
{
    "topic":"pizza"
}
```

Aur next node tha:

```
generate_joke
```

Ab jab invoke kiya gaya,

Workflow ne:

```
Pizza

↓

Generate Joke

↓

Generate Explanation

↓

END
```

execute kiya.

Output me ek naya joke aur uski explanation generate ho gayi.

---

### `workflow.get_state_history(config1)`

```python
list(workflow.get_state_history(config1))
```

Ab history dobara dekhi gayi.

Ab history me do executions dikhengi.

Example:

```
Execution 1

Pizza

↓

Old Joke

↓

Old Explanation
```

Aur Time Travel ke baad

```
Execution 2

Pizza

↓

New Joke

↓

New Explanation
```

Matlab purane checkpoints delete nahi hue.

Unke upar ek naya execution path create ho gaya.

---

### Time Travel ka Real Use

Time Travel ka use tab hota hai jab:

- Kisi specific step se workflow dobara chalana ho.
- Debugging karni ho.
- Different outputs compare karne ho.
- Kisi purani state ko restore karna ho.
- Same input se multiple responses generate karne ho.

Example:

Pehli baar:

```
Pizza Joke 1
```

Time Travel ke baad:

```
Pizza Joke 2
```

Ab dono outputs compare kiye ja sakte hain.

---

### Summary

- `checkpoint_id` workflow ke kisi saved checkpoint ko identify karta hai.
- `get_state()` us checkpoint ki state retrieve karta hai.
- `next` batata hai workflow ka next executable node kaunsa hai.
- `invoke(None, config)` workflow ko usi checkpoint se continue karta hai bina nayi input diye.
- `get_state_history()` execution ke baad updated history dikhata hai.
- Time Travel debugging aur workflow replay ke liye LangGraph ka bahut powerful feature hai.

In [110]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:56:38.565188+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7230-65a8-bfff-0a96c2fc4e11'}}, tasks=(PregelTask(id='dcd96e38-1f32-5ed6-9f44-fa2b22c193f0', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza go to the doctor? Because it was feeling a little saucy!'}),), interrupts=())

In [111]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

{'topic': 'pizza',
 'joke': 'Why did the mushroom go to the pizza party? Because he was a fungi and everyone wanted a pizza him!',
 'explanation': 'This joke plays on the word "fun guy" (fungi) which sounds like "fungi," a type of mushroom. The play on words is that the mushroom went to the pizza party because he was a "fun guy" and people wanted to "pizza" (see) him. The joke is a pun that combines the idea of mushrooms being fungi with the concept of being a fun person at a party.'}

In [112]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the mushroom go to the pizza party? Because he was a fungi and everyone wanted a pizza him!', 'explanation': 'This joke plays on the word "fun guy" (fungi) which sounds like "fungi," a type of mushroom. The play on words is that the mushroom went to the pizza party because he was a "fun guy" and people wanted to "pizza" (see) him. The joke is a pun that combines the idea of mushrooms being fungi with the concept of being a fun person at a party.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc70-100a-6bff-8002-7d6c3d37b1f4'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:57:21.959833+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc70-064c-630b-8001-707d60a085ad'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the mushroom go to the 

#### Updating State

### Updating Workflow State

Is cell me hum **LangGraph ke `update_state()`** method ka use kar rahe hain. Is method ki help se hum kisi existing workflow ke **saved checkpoint ki state ko manually update** kar sakte hain, bina workflow ko start se dobara run kiye.

Ye feature debugging, testing aur workflow ko kisi specific point se naye data ke saath continue karne ke liye bahut useful hota hai.

---

### `workflow.update_state()`

```python
workflow.update_state(config, {"topic":"samosa"})
```

`update_state()` ka kaam kisi existing checkpoint ki state me naye values update karna hota hai.

Is example me humne:

```python
{
    "topic":"pizza"
}
```

ko update karke

```python
{
    "topic":"samosa"
}
```

kar diya.

Dhyan rahe ki yaha workflow dobara execute nahi hua hai, sirf stored state change hui hai.

---

### Parameters

#### Configuration

```python
{
    "thread_id":"1",
    "checkpoint_id":"..."
}
```

Ye batata hai ki kis thread aur kis checkpoint ki state update karni hai.

#### Updated State

```python
{
    "topic":"samosa"
}
```

Ye nayi value hai jo existing state me update ki ja rahi hai.

---

### Update hone ke baad kya hota hai?

Jab state update hoti hai, LangGraph ek **naya checkpoint** create karta hai.

Purana checkpoint delete nahi hota.

Matlab history kuch is tarah ban jaati hai:

```
Checkpoint 1

Topic = Pizza
```

↓

```
Checkpoint 2

Topic = Samosa
```

Isliye workflow ki purani history bhi safe rehti hai.

---

### `workflow.get_state_history(config1)`

```python
list(workflow.get_state_history(config1))
```

State update karne ke baad history check ki gayi.

Ab history me ek naya checkpoint dikhai dega jisme

```python
topic = "samosa"
```

store hoga.

Isse verify hota hai ki update successfully ho gaya hai.

---

### `workflow.invoke(None, config)`

```python
workflow.invoke(
    None,
    {
        "configurable":{
            "thread_id":"1",
            "checkpoint_id":"..."
        }
    }
)
```

Yaha `None` pass kiya gaya hai.

Iska matlab:

> Nayi input mat do. Updated checkpoint se workflow ko continue karo.

Ab kyunki topic update hokar

```
samosa
```

ho chuka hai, isliye workflow pizza ki jagah samosa ke upar joke aur explanation generate karega.

Execution Flow:

```
Updated State

Topic = Samosa

↓

Generate Joke

↓

Generate Explanation

↓

END
```

---

### Generated Output

Ab output kuch is type ka milega:

```python
{
    "topic":"samosa",
    "joke":"...",
    "explanation":"..."
}
```

Yani workflow ne updated state ke basis par completely naya result generate kiya.

---

### Final State History

Dobara

```python
list(workflow.get_state_history(config1))
```

chalane par history me dono executions dikhte hain.

Example:

```
Old Execution

Topic = Pizza
```

↓

```
Updated Checkpoint

Topic = Samosa
```

↓

```
New Joke

↓

New Explanation
```

Matlab workflow ki history preserve rehti hai aur naye execution ke checkpoints usme add ho jaate hain.

---

### Real World Use Cases

`update_state()` ka use bahut useful hota hai jab:

- Kisi user ka input manually correct karna ho.
- Workflow ko naye data ke saath continue karna ho.
- Debugging ke time kisi variable ki value change karni ho.
- Kisi specific checkpoint se alternate execution generate karna ho.

Example:

Pehle:

```
Topic = Pizza
```

Developer ne state update ki:

```
Topic = Samosa
```

Ab workflow same checkpoint se continue hua aur samosa ke upar naya joke generate ho gaya.

---

### Summary

- `update_state()` existing checkpoint ki state ko modify karta hai.
- State update hone ke baad LangGraph ek naya checkpoint create karta hai.
- Purani history delete nahi hoti.
- `get_state_history()` se updated checkpoints dekhe ja sakte hain.
- `invoke(None, config)` updated checkpoint se workflow ko continue karta hai.
- Ye feature debugging, testing aur state modification ke liye bahut powerful hai.

In [113]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f06cc72-ca16-6359-8001-7eea05e07dd2'}}

In [114]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc72-ca16-6359-8001-7eea05e07dd2'}}, metadata={'source': 'update', 'step': 1, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:58:35.155132+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='0f085bb0-c1e8-d9fd-fb15-c427126b7cd6', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the mushroom go to the pizza party? Because he was a fungi and everyone wanted a pizza him!', 'explanation': 'This joke plays on the word "fun guy" (fungi) which sounds like "fungi," a type of mushroom. The play on words is that the mushroom went to the pizza party because he was a "fun guy" and people 

In [115]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc72-ca16-6359-8001-7eea05e07dd2"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa bring a ladder to the party? \nBecause it wanted to be the best snack in the room and rise to the occasion!',
 'explanation': 'This joke plays on the double meaning of the word "rise." In one sense, "rise" means to physically move upwards, which is why the samosa brought a ladder to the party. However, in another sense, "rise" can also mean to perform well or excel, as in rising to the occasion. So, the samosa brought a ladder to symbolize its desire to physically rise above the other snacks at the party and also to metaphorically rise to the occasion by being the best snack in the room.'}

In [116]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa bring a ladder to the party? \nBecause it wanted to be the best snack in the room and rise to the occasion!', 'explanation': 'This joke plays on the double meaning of the word "rise." In one sense, "rise" means to physically move upwards, which is why the samosa brought a ladder to the party. However, in another sense, "rise" can also mean to perform well or excel, as in rising to the occasion. So, the samosa brought a ladder to symbolize its desire to physically rise above the other snacks at the party and also to metaphorically rise to the occasion by being the best snack in the room.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc75-4407-6195-8003-b08dcfd27511'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:59:41.628661+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoin

### Fault Tolerance

In [2]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [3]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [4]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [5]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))